# 16 — Concurrency: Threads, Processes, and `concurrent.futures`

Goal: understand the GIL, pick the right concurrency model, and avoid common race conditions.

_Generated: 2026-02-19_

## Setup

This course targets **Python 3.11+** (works on 3.10+, with a few feature differences).

Recommended tooling:

```bash
# create + activate a virtual environment
python -m venv .venv
# mac/linux:
source .venv/bin/activate
# windows (PowerShell):
# .venv\Scripts\Activate.ps1

python -m pip install -U pip

# quality-of-life (optional but recommended)
python -m pip install -U ipykernel ruff black pytest mypy
```

If you're using Jupyter:
```bash
python -m ipykernel install --user --name python-course --display-name "Python Course (.venv)"
```

In [ ]:

import sys, platform, os
print("python:", sys.version.split()[0])
print("implementation:", platform.python_implementation())
print("platform:", platform.platform())
print("cwd:", os.getcwd())


## 1.
L1: The GIL (Global Interpreter Lock) in one paragraph

In CPython, only one thread executes Python bytecode at a time.
This means:
- Threads help **I/O-bound** tasks (waiting on network/disk).
- Processes help **CPU-bound** tasks (parallel CPU).

## 2.
L2: Threads with `ThreadPoolExecutor` (I/O-bound)

We’ll simulate I/O with `time.sleep`.

In [ ]:

import time
from concurrent.futures import ThreadPoolExecutor, as_completed

def fake_io(i: int) -> str:
    time.sleep(0.05)
    return f"done {i}"

t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=8) as ex:
    futs = [ex.submit(fake_io, i) for i in range(20)]
    results = [f.result() for f in as_completed(futs)]
dt = time.perf_counter() - t0
print("completed:", len(results), "in", round(dt, 3), "sec")


## 3.
L3: Processes with `ProcessPoolExecutor` (CPU-bound)

Warning: in notebooks and on Windows/macOS, multiprocessing can behave differently.
In production scripts, always guard with `if __name__ == "__main__":`.

In [ ]:

from concurrent.futures import ProcessPoolExecutor
import math

def cpu_work(n: int) -> int:
    # a tiny bit of CPU work
    s = 0
    for i in range(1, n):
        s += int(math.sqrt(i))
    return s

# Keep small for notebook safety
with ProcessPoolExecutor(max_workers=2) as ex:
    out = list(ex.map(cpu_work, [20_000, 20_000, 20_000, 20_000]))
print(out[:2], "...")


## 4.
L4: Shared state, locks, and race conditions

If multiple threads modify shared state, you need synchronization.
Prefer avoiding shared mutable state.

In [ ]:

import threading

counter = 0
lock = threading.Lock()

def inc(n: int):
    global counter
    for _ in range(n):
        with lock:
            counter += 1

threads = [threading.Thread(target=inc, args=(10_000,)) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()

print("counter:", counter)


## 5.
L5: Queues (producer/consumer)

`queue.Queue` is thread-safe and great for pipelines.

In [ ]:

import queue, threading, time

q: queue.Queue[int | None] = queue.Queue()
out: list[int] = []

def producer():
    for i in range(5):
        q.put(i)
    q.put(None)  # sentinel

def consumer():
    while True:
        item = q.get()
        if item is None:
            break
        out.append(item * 2)

tp = threading.Thread(target=producer)
tc = threading.Thread(target=consumer)
tp.start(); tc.start()
tp.join(); tc.join()

print(out)


## 6.
L6: Exercises

1. Use `ThreadPoolExecutor` to fetch multiple URLs (optional: with `requests`).
2. Demonstrate a race condition by removing the lock from the counter example.
3. Write a producer/consumer that squares numbers.

## 7.
L7: Common pitfalls

Threads:
- race conditions on shared state
- deadlocks with multiple locks
- blocking I/O without timeouts

Processes:
- functions/args must be pickleable
- overhead of process startup + IPC
- guard `if __name__ == "__main__"` in scripts (especially on Windows/macOS)

## 8.
L8: Concurrency primitives quick reference

- `threading.Lock`, `RLock`
- `threading.Event`
- `threading.Semaphore`
- `queue.Queue` (thread-safe)
- `multiprocessing.Queue` (process-safe)

Prefer high-level executors unless you need fine control.